# jPCA v2 — Corrected Rotational Dynamics Analysis

Fixes vs `03_jpca.ipynb`:
1. **Noise at eval** — `noise_at_eval=True` to match biological trial-to-trial variability
2. **Paper-style permutation test** — shuffles conditions per neuron (not time within condition)
3. **Loads v2 checkpoint** (if available, falls back to v1)
4. **Compares both permutation methods** side by side

In [ ]:
import sys

sys.path.insert(0, "..")

import numpy as np
import torch
import matplotlib.pyplot as plt
from pathlib import Path

from src.model import BioLeakyRNN
from src.env import CuedTargetWithDistractorsV3
from src.analysis import (
    collect_trials,
    prepare_jpca_input,
    fit_jpca,
    jpca_permutation_test,
    jpca_permutation_test_paper,
)
from src.plotting import (
    plot_jpca_trajectories,
    plot_jpca_timecourses,
    plot_jpca_permutation_test,
)

device = "cpu"
print("device:", device)

## 1. Load model

Uses v2 checkpoint if available, otherwise falls back to v1.

In [ ]:
# Choose checkpoint: prefer v2, fall back to v1
ckpt_v2 = Path("../checkpoints/stage2_v2.pt")
ckpt_v1 = Path("../checkpoints/stage2.pt")

if ckpt_v2.exists():
    ckpt_path = ckpt_v2
    sigma_rec = 0.10  # v2 was trained with higher noise
    print("Using v2 checkpoint")
else:
    ckpt_path = ckpt_v1
    sigma_rec = 0.05  # v1 original
    print("v2 checkpoint not found, using v1")


def make_model():
    return BioLeakyRNN(
        input_size=7,
        hidden_size=128,
        output_size=2,
        dt=20.0,
        tau=100.0,
        activation="softplus",
        sigma_rec=sigma_rec,
        use_ei=True,
        exc_ratio=0.7,
        use_dale=True,
        mask_seed=42,
    )


def make_env():
    return CuedTargetWithDistractorsV3(
        dt=20, cue_strength=1.0, p_distractor_trial=0.6, distractor_strength=1.0
    )


model = make_model().to(device)
model.load_state_dict(torch.load(str(ckpt_path), weights_only=True)["state_dict"])
model.eval()
model.noise_at_eval = True  # <-- key fix: inject noise during analysis
print(f"Loaded {ckpt_path.name}, noise_at_eval={model.noise_at_eval}")

## 2. Collect trials (with noise)

In [ ]:
print("Collecting 5000 trials (with recurrent noise)...")
trials = collect_trials(model, make_env, n_trials=5000, device=device)

outcomes = {}
for tr in trials:
    outcomes[tr["train_outcome"]] = outcomes.get(tr["train_outcome"], 0) + 1
total = len(trials)
print(f"Total: {total}")
for o, n in sorted(outcomes.items(), key=lambda x: -x[1]):
    print(f"  {o}: {n} ({100*n/total:.1f}%)")

## 3. Build condition-averaged input

In [ ]:
X_cond, labels, rel_time, counts = prepare_jpca_input(
    trials,
    align_key="target_onset",
    window_before=15,  # 300 ms
    window_after=30,  # 600 ms
    min_trials=10,
    outcome="correct",
)

print(f"X_cond shape: {X_cond.shape}")
print(f"CTOA bins: {labels}")
print(f"Trials per bin: {counts}")

## 4. Fit jPCA

In [ ]:
jpca_res = fit_jpca(X_cond, n_jpcs=1, pca_dims=6)

print("=== jPCA results (with noise at eval) ===")
print(f"var_ratio  : {jpca_res['var_ratio']:.3f}   (paper: M1=0.79, M2=0.60)")
print(f"rot_index  : {jpca_res['rot_index']:.3f}   (paper: M1=0.70, M2=0.47)")
print()

eigs = jpca_res["eigenvalues"]
print("Top eigenvalues of M_skew:")
for i in range(min(4, len(eigs))):
    print(f"  lam{i+1} = {eigs[i].real:+.4f} {eigs[i].imag:+.4f}i")

evr = jpca_res["pca_pre"].explained_variance_ratio_
print(f"\nPre-PCA variance (6 dims): {evr.sum()*100:.1f}%")
print("Per component:", np.round(evr * 100, 1))

# jPC variance
X_flat = X_cond.reshape(-1, X_cond.shape[-1])
total_var = np.var(X_flat, axis=0).sum()
Z_flat = jpca_res["Z"].reshape(-1, jpca_res["Z"].shape[-1])
jpc_var = np.var(Z_flat, axis=0)
print(f"\njPC1 var : {jpc_var[0]/total_var*100:.1f}%")
print(f"jPC2 var : {jpc_var[1]/total_var*100:.1f}%")
print(f"jPC1+2   : {jpc_var[:2].sum()/total_var*100:.1f}%   (paper: ~46%)")

## 5. Visualise

In [ ]:
plot_jpca_trajectories(
    jpca_res,
    labels=labels,
    rel_time=rel_time,
    align_label="target_onset",
    cmap="plasma",
    title="jPCA v2 — CTOA bins (noise at eval)",
)

In [ ]:
plot_jpca_timecourses(
    jpca_res,
    labels=labels,
    rel_time=rel_time,
    align_label="target_onset",
    cmap="plasma",
)

## 6. Permutation tests — BOTH methods

### 6a. Original method (shuffle time within condition)

In [ ]:
print("Permutation test — time shuffle (original method)...")
perm_time = jpca_permutation_test(
    X_cond,
    n_perms=100,
    pca_dims=6,
    n_jpcs=1,
    seed=42,
)
print(
    f"  var_ratio: obs={perm_time['real_var_ratio']:.3f}  "
    f"null={perm_time['perm_var_ratios'].mean():.3f}+/-{perm_time['perm_var_ratios'].std():.3f}  "
    f"p={perm_time['p_var_ratio']:.3f}"
)
print(
    f"  rot_index: obs={perm_time['real_rot_index']:.3f}  "
    f"null={perm_time['perm_rot_indices'].mean():.3f}+/-{perm_time['perm_rot_indices'].std():.3f}  "
    f"p={perm_time['p_rot_index']:.3f}"
)

### 6b. Paper method (shuffle conditions per neuron)

In [ ]:
print("Permutation test — condition shuffle per neuron (paper method)...")
perm_paper = jpca_permutation_test_paper(
    X_cond,
    n_perms=100,
    pca_dims=6,
    n_jpcs=1,
    seed=42,
)
print(
    f"  var_ratio: obs={perm_paper['real_var_ratio']:.3f}  "
    f"null={perm_paper['perm_var_ratios'].mean():.3f}+/-{perm_paper['perm_var_ratios'].std():.3f}  "
    f"p={perm_paper['p_var_ratio']:.3f}"
)
print(
    f"  rot_index: obs={perm_paper['real_rot_index']:.3f}  "
    f"null={perm_paper['perm_rot_indices'].mean():.3f}+/-{perm_paper['perm_rot_indices'].std():.3f}  "
    f"p={perm_paper['p_rot_index']:.3f}"
)

### 6c. Compare both null distributions

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# var_ratio
ax = axes[0]
ax.hist(
    perm_time["perm_var_ratios"],
    bins=20,
    alpha=0.5,
    color="steelblue",
    label=f"Time shuffle (mean={perm_time['perm_var_ratios'].mean():.3f})",
)
ax.hist(
    perm_paper["perm_var_ratios"],
    bins=20,
    alpha=0.5,
    color="coral",
    label=f"Condition shuffle (mean={perm_paper['perm_var_ratios'].mean():.3f})",
)
ax.axvline(
    perm_time["real_var_ratio"],
    color="k",
    ls="--",
    lw=2,
    label=f"Observed = {perm_time['real_var_ratio']:.3f}",
)
ax.set_xlabel("var_ratio")
ax.set_ylabel("count")
ax.set_title("Rotational Variance Ratio")
ax.legend(fontsize=9)

# rot_index
ax = axes[1]
ax.hist(
    perm_time["perm_rot_indices"],
    bins=20,
    alpha=0.5,
    color="steelblue",
    label=f"Time shuffle (mean={perm_time['perm_rot_indices'].mean():.3f})",
)
ax.hist(
    perm_paper["perm_rot_indices"],
    bins=20,
    alpha=0.5,
    color="coral",
    label=f"Condition shuffle (mean={perm_paper['perm_rot_indices'].mean():.3f})",
)
ax.axvline(
    perm_time["real_rot_index"],
    color="k",
    ls="--",
    lw=2,
    label=f"Observed = {perm_time['real_rot_index']:.3f}",
)
ax.set_xlabel("rot_index")
ax.set_ylabel("count")
ax.set_title("Rotational Index")
ax.legend(fontsize=9)

plt.suptitle("Permutation Tests: Time Shuffle vs Condition Shuffle (paper)", fontsize=13)
plt.tight_layout()
plt.show()

## 7. Summary comparison

| Metric | Paper (M1/M2) | v1 (no noise) | v2 (noise at eval) |
|--------|---------------|---------------|--------------------|
| var_ratio | 0.79 / 0.60 | 0.728 | ? |
| rot_index | 0.70 / 0.47 | 0.869 | ? |
| jPC1+2 var | 46% | 14.2% | ? |
| p (var_ratio, paper-style) | <0.05 | N/A | ? |
| p (rot_index, paper-style) | <0.05 | N/A | ? |